In [2]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from collections import deque
import sys
import time


In [3]:
# Cell 2: HybridPowerFlowOptimizer Class (Aligned with Paper's Objective)
class HybridPowerFlowOptimizer:
    """
    Optimizes generator rescheduling to manage congestion by minimizing rescheduling cost,
    based on incremental/decremental bids. Uses a SLACK BUS for power balance.
    Loads are fixed (no load shedding). Aligned with the objective in the reference paper.
    """

    # MODIFIED __init__ method: Use Inc/Dec Costs, Fix Loads
    def __init__(self, A_matrix, line_limits,
                 gen_costs_inc_full, gen_costs_dec_full, # <-- NEW: Separate Inc/Dec Costs
                 gen_limits_min_full, gen_limits_max_full,
                 initial_B_net, fixed_load_full,
                 gen_indices, load_indices,
                 slack_bus_index,
                 memory_size=500):
        """
        Initializes the optimizer for minimum rescheduling cost.

        Args:
            A_matrix (np.ndarray): System matrix (e.g., PTDF).
            line_limits (np.ndarray): Absolute power flow limits for each line.
            gen_costs_inc_full (np.ndarray): Incremental cost bids ($/MW inc) for ALL buses.
            gen_costs_dec_full (np.ndarray): Decremental cost bids ($/MW dec) for ALL buses.
            gen_limits_min_full (np.ndarray): Min generation (Pg) limit for ALL buses.
            gen_limits_max_full (np.ndarray): Max generation (Pg) limit for ALL buses.
            initial_B_net (np.ndarray): Initial NET bus injections (Pg - Pl).
            fixed_load_full (np.ndarray): Fixed load demand (Pl >= 0) for ALL buses.
            gen_indices (list or np.ndarray): 0-based indices for generator buses.
            load_indices (list or np.ndarray): 0-based indices for fixed load buses.
            slack_bus_index (int): 0-based index of the designated slack bus.
            memory_size (int): Size of the memory for storing past feasible solutions.
        """
        self.A = A_matrix
        self.line_limits = np.array(line_limits, dtype=np.float64)
        self.num_lines = A_matrix.shape[0]
        self.num_buses = A_matrix.shape[1]
        self.tolerance = 1e-6

        # --- Store Initial Net Injection, Fixed Load, and Indices ---
        self.initial_B_net = initial_B_net.copy().flatten()
        self.fixed_load = fixed_load_full.copy().flatten()
        if np.any(self.fixed_load < 0):
            print("Warning: Fixed loads should be non-negative. Clamping negative loads to zero.")
            self.fixed_load = np.maximum(0, self.fixed_load)

        self.gen_indices = np.array(sorted(gen_indices), dtype=int)
        self.load_indices = np.array(sorted(load_indices), dtype=int) # Still needed to identify non-gens

        # --- Slack Bus Validation ---
        self.slack_bus_index = int(slack_bus_index)
        if len(self.gen_indices) == 0:
             raise ValueError("Cannot assign a slack bus when no generators are specified.")
        if self.slack_bus_index not in self.gen_indices:
            raise ValueError(f"Specified Slack Bus index ({self.slack_bus_index}) is not in the list of Generator indices ({self.gen_indices}).")
        print(f"Using Bus {self.slack_bus_index+1} as the Slack Bus.")
        self.non_slack_gen_indices = np.setdiff1d(self.gen_indices, [self.slack_bus_index], assume_unique=True)

        # --- Bus Type Validation ---
        all_indices = np.concatenate((self.gen_indices, self.load_indices))
        if len(np.unique(all_indices)) != self.num_buses or len(all_indices) != self.num_buses:
             raise ValueError("Generator and Load indices do not form a complete, non-overlapping set of all buses.")
        print(f"Using specified {len(self.gen_indices)} generator buses (indices: {self.gen_indices})")
        print(f"Using specified {len(self.load_indices)} fixed load buses (indices: {self.load_indices})")

        # --- Store Generator-Specific Data (Limits and Inc/Dec Costs) ---
        num_gens = len(self.gen_indices)
        self.gen_costs_inc_only = np.zeros(num_gens) # Incremental costs for generators
        self.gen_costs_dec_only = np.zeros(num_gens) # Decremental costs for generators
        self.gen_limits_Pg_min_only = np.zeros(num_gens)
        self.gen_limits_Pg_max_only = np.zeros(num_gens)

        # Find local index of slack bus within gen_indices array
        self.slack_bus_local_gen_idx_ = np.where(self.gen_indices == self.slack_bus_index)[0]
        if len(self.slack_bus_local_gen_idx_) == 0: raise ValueError("Internal Error: Could not find slack bus local index.")
        self.slack_bus_local_gen_idx = self.slack_bus_local_gen_idx_[0]

        if num_gens > 0:
            # Validate and extract costs and limits
            flat_gc_inc = np.array(gen_costs_inc_full).flatten()
            flat_gc_dec = np.array(gen_costs_dec_full).flatten()
            flat_gmin = np.array(gen_limits_min_full).flatten()
            flat_gmax = np.array(gen_limits_max_full).flatten()
            if len(flat_gc_inc)!=self.num_buses or len(flat_gc_dec)!=self.num_buses or \
               len(flat_gmin)!=self.num_buses or len(flat_gmax)!=self.num_buses:
                raise ValueError(f"Generator cost/limit array length mismatch (Expected {self.num_buses})")

            self.gen_costs_inc_only = flat_gc_inc[self.gen_indices]
            self.gen_costs_dec_only = flat_gc_dec[self.gen_indices]
            self.gen_limits_Pg_min_only = flat_gmin[self.gen_indices]
            self.gen_limits_Pg_max_only = flat_gmax[self.gen_indices]

            if np.any(self.gen_limits_Pg_max_only < self.gen_limits_Pg_min_only):
                raise ValueError("Generator Max Pg limit cannot be less than Min Pg limit.")
            if np.any(self.gen_costs_inc_only < 0) or np.any(self.gen_costs_dec_only < 0):
                 print("Warning: Generator cost bids should ideally be non-negative.")

        # --- Calculate and Store NET Injection Limits (Bnet = Pg - Pl) ---
        # For Generators:
        self.gen_limits_Bnet_min_only = np.zeros(num_gens)
        self.gen_limits_Bnet_max_only = np.zeros(num_gens)
        if num_gens > 0:
            fixed_load_at_gens = self.fixed_load[self.gen_indices]
            self.gen_limits_Bnet_min_only = self.gen_limits_Pg_min_only - fixed_load_at_gens
            self.gen_limits_Bnet_max_only = self.gen_limits_Pg_max_only - fixed_load_at_gens

        # --- NO Load Bus Limits needed as they are fixed ---
        # B_net for load buses is fixed at -Pl
        # We need to ensure the initial state reflects this for load buses
        self.initial_B_net[self.load_indices] = -self.fixed_load[self.load_indices]
        print(f"-> Fixed B_net for load buses set to -Pl.")


        # --- Adjust Initial NET Injection State if Necessary ---
        needs_adjust = False
        # 1. Clamp initial NET injection at ALL generator buses if implied Pg is outside limits
        if num_gens > 0:
            initial_Bnet_at_gens = self.initial_B_net[self.gen_indices]
            clipped_Bnet_gens = np.clip(initial_Bnet_at_gens,
                                        self.gen_limits_Bnet_min_only,
                                        self.gen_limits_Bnet_max_only)
            if np.any(np.abs(initial_Bnet_at_gens - clipped_Bnet_gens) > self.tolerance):
                print("Warning: Initial net injection at generator bus(es) implies Pg outside limits. Clamping net injection...")
                self.initial_B_net[self.gen_indices] = clipped_Bnet_gens
                needs_adjust = True

        # 2. Ensure load bus B_net is exactly -Pl (already done above, but double-check)
        target_load_bnet = -self.fixed_load[self.load_indices]
        if np.any(np.abs(self.initial_B_net[self.load_indices] - target_load_bnet) > self.tolerance):
             print("Warning: Overwriting initial B_net for load buses to ensure it matches -Pl.")
             self.initial_B_net[self.load_indices] = target_load_bnet
             needs_adjust = True

        # 3. Balance initial NET injections using ONLY THE SLACK BUS
        required_total_injection = 0.0
        current_total_injection = np.sum(self.initial_B_net)
        difference_init = required_total_injection - current_total_injection

        if abs(difference_init) > self.tolerance * self.num_buses:
            print(f"Warning: Initial NET injections sum to {current_total_injection:.4f} (≠ 0). Adjusting SLACK BUS net injection to balance...")
            adjusted_Bnet_slack = self.initial_B_net[self.slack_bus_index] + difference_init
            slack_min_Bnet = self.gen_limits_Bnet_min_only[self.slack_bus_local_gen_idx]
            slack_max_Bnet = self.gen_limits_Bnet_max_only[self.slack_bus_local_gen_idx]
            clipped_adjusted_Bnet_slack = np.clip(adjusted_Bnet_slack, slack_min_Bnet, slack_max_Bnet)
            actual_adjustment_applied = clipped_adjusted_Bnet_slack - self.initial_B_net[self.slack_bus_index]
            self.initial_B_net[self.slack_bus_index] = clipped_adjusted_Bnet_slack
            remaining_diff = difference_init - actual_adjustment_applied
            if abs(remaining_diff) > self.tolerance:
                print(f"Warning: Could not fully balance initial state due to SLACK BUS net injection limits. Remaining imbalance: {remaining_diff:.4f}")
            needs_adjust = True

        if needs_adjust:
            print("-> Adjusted initial Net Injection (B) state used for optimization:", np.round(self.initial_B_net, 4))
        # --- End Initial State Handling ---

        self.memory = deque(maxlen=memory_size)
        self.stagnation_threshold = 50
        self.diversification_fraction = 0.2

    # MODIFIED _apply_constraints method: Fix Loads, Use Slack Bus
    def _apply_constraints(self, solution_vector):
        """
        Applies constraints: Fixes load bus B_net, clips non-slack generator B_net,
        and adjusts slack bus B_net to enforce power balance (sum(B_net) = 0).
        Slack bus is NOT clipped here.
        """
        sol_Bnet = solution_vector.copy().flatten()
        num_non_slack_gens = len(self.non_slack_gen_indices)

        # 1. Ensure Load Bus B_net remains fixed at -Pl
        # (This prevents the optimization from trying to change loads)
        sol_Bnet[self.load_indices] = -self.fixed_load[self.load_indices]

        # 2. Apply NON-SLACK Generator Bus Net Injection Limits
        if num_non_slack_gens > 0:
             non_slack_gen_local_indices = np.where(np.isin(self.gen_indices, self.non_slack_gen_indices))[0]
             sol_Bnet[self.non_slack_gen_indices] = np.clip(sol_Bnet[self.non_slack_gen_indices],
                                                             self.gen_limits_Bnet_min_only[non_slack_gen_local_indices],
                                                             self.gen_limits_Bnet_max_only[non_slack_gen_local_indices])

        # 3. Enforce Power Balance (Adjust ONLY SLACK BUS Net Injection)
        required_total_Bnet = 0.0
        # Calculate sum excluding the slack bus first, as its value will be determined
        non_slack_indices = np.concatenate((self.non_slack_gen_indices, self.load_indices))
        current_sum_excluding_slack = np.sum(sol_Bnet[non_slack_indices])
        # Required slack injection = 0 - sum(other injections)
        required_slack_bnet = required_total_Bnet - current_sum_excluding_slack
        sol_Bnet[self.slack_bus_index] = required_slack_bnet

        # --- NO CLIPPING of slack bus here in _apply_constraints ---
        # Limits are handled via penalty in fitness function.

        return sol_Bnet.reshape(-1, 1)


    # MODIFIED _calculate_fitness method: Minimize Rescheduling Cost + Penalties
    def _calculate_fitness(self, solution_Bnet):
        """
        Calculates fitness = Rescheduling Cost + Penalties for constraint violations.
        Objective is to minimize the cost of rescheduling generators.
        """
        sol_Bnet_flat = solution_Bnet.flatten()
        if not np.all(np.isfinite(sol_Bnet_flat)): return np.inf

        # Calculate line flows
        try:
            C = np.dot(self.A, sol_Bnet_flat); flows = C.flatten()
            if not np.all(np.isfinite(C)): return np.inf
        except ValueError: return np.inf

        # --- Penalty Components ---
        penalty_multiplier = 1e10 # High penalty for constraint violations

        # 1. Line Limit Penalty
        line_violations = np.maximum(0, np.abs(flows) - (self.line_limits + self.tolerance))
        line_violation_penalty = penalty_multiplier * np.sum(line_violations**2)

        # 2. NON-SLACK Generator Net Injection Limit Penalty
        gen_limit_penalty = 0.0
        num_non_slack_gens = len(self.non_slack_gen_indices)
        if num_non_slack_gens > 0:
            non_slack_gen_local_indices = np.where(np.isin(self.gen_indices, self.non_slack_gen_indices))[0]
            non_slack_gen_Bnet_values = sol_Bnet_flat[self.non_slack_gen_indices]
            violations_lower_g = np.maximum(0, self.gen_limits_Bnet_min_only[non_slack_gen_local_indices] - non_slack_gen_Bnet_values + self.tolerance)
            violations_upper_g = np.maximum(0, non_slack_gen_Bnet_values - self.gen_limits_Bnet_max_only[non_slack_gen_local_indices] - self.tolerance)
            gen_limit_penalty = penalty_multiplier * (np.sum(violations_lower_g**2) + np.sum(violations_upper_g**2))

        # 3. SLACK BUS Net Injection Limit Penalty
        slack_limit_penalty = 0.0
        slack_Bnet_value = sol_Bnet_flat[self.slack_bus_index]
        slack_min_Bnet = self.gen_limits_Bnet_min_only[self.slack_bus_local_gen_idx]
        slack_max_Bnet = self.gen_limits_Bnet_max_only[self.slack_bus_local_gen_idx]
        violation_lower_s = np.maximum(0, slack_min_Bnet - slack_Bnet_value + self.tolerance)
        violation_upper_s = np.maximum(0, slack_Bnet_value - slack_max_Bnet - self.tolerance)
        slack_limit_penalty = penalty_multiplier * (violation_lower_s**2 + violation_upper_s**2)

        # 4. Load-Only Net Injection Limit Penalty - REMOVED (Loads are fixed)

        # 5. Power Balance Penalty (Keep for robustness)
        balance_violation = abs(np.sum(sol_Bnet_flat))
        balance_penalty = penalty_multiplier * (balance_violation**2) if balance_violation > self.tolerance * 10 else 0.0

        # --- Objective Component ---
        # Minimize the total rescheduling cost based on incremental/decremental bids
        rescheduling_cost = self._get_rescheduling_cost(solution_Bnet) # Uses inc/dec costs

        # Total Fitness = Objective + Penalties
        fitness = (rescheduling_cost + # Objective term (weight = 1.0)
                   line_violation_penalty + gen_limit_penalty + slack_limit_penalty +
                   balance_penalty)

        return fitness if np.isfinite(fitness) else np.inf

    # MODIFIED _get_rescheduling_cost: Use Incremental/Decremental Costs
    def _get_rescheduling_cost(self, solution_Bnet):
        """
        Calculates the total rescheduling cost based on incremental/decremental bids
        for ALL generators, according to the change from the initial state.
        Cost = sum( C_inc * delta_P+ + C_dec * delta_P- )
        """
        sol_Bnet_flat = solution_Bnet.flatten()
        if len(self.gen_indices) == 0: return 0.0
        cost = 0.0
        try:
            # Calculate change in B_net (which equals change in Pg for generators)
            delta_Bnet = sol_Bnet_flat[self.gen_indices] - self.initial_B_net[self.gen_indices]

            # Calculate cost for each generator based on the sign of the change
            for i in range(len(self.gen_indices)):
                delta = delta_Bnet[i]
                if delta > self.tolerance: # Increase in generation (Positive delta_Pg)
                    cost += self.gen_costs_inc_only[i] * delta
                elif delta < -self.tolerance: # Decrease in generation (Negative delta_Pg)
                    cost += self.gen_costs_dec_only[i] * abs(delta) # Use decremental cost bid

            return cost if np.isfinite(cost) else np.inf
        except IndexError:
            print("Warning (_get_rescheduling_cost): IndexError.")
            return np.inf
        except Exception as e:
             print(f"Warning (_get_rescheduling_cost): Error {e}")
             return np.inf

    # MODIFIED _is_feasible method: Remove Load Limit Check
    def _is_feasible(self, solution_Bnet, verbose=False):
        """
        Checks if a NET injection solution meets all hard constraints.
        Loads are fixed, so no check needed for them.
        Checks: Line limits, non-slack generator limits, slack generator limits, power balance.
        """
        if solution_Bnet is None:
            if verbose: print("DEBUG (_is_feasible): Input solution is None.")
            return False
        sol_Bnet_flat = solution_Bnet.flatten()
        if not np.all(np.isfinite(sol_Bnet_flat)):
            if verbose: print("DEBUG (_is_feasible): Solution contains non-finite values.")
            return False

        # Check Line Limits
        line_ok = False; flows = np.array([])
        try:
            C = np.dot(self.A, sol_Bnet_flat); flows = C.flatten()
            if not np.all(np.isfinite(C)): raise ValueError("Flows NaN/Inf")
            line_ok = np.all(np.abs(flows) <= self.line_limits + self.tolerance)
        except Exception as e: line_ok = False;

        # Check NON-SLACK Generator Net Injection Limits
        non_slack_gen_ok = True
        if len(self.non_slack_gen_indices) > 0:
            non_slack_gen_local_indices = np.where(np.isin(self.gen_indices, self.non_slack_gen_indices))[0]
            non_slack_gen_Bnet_values = sol_Bnet_flat[self.non_slack_gen_indices]
            non_slack_gen_ok = np.all(non_slack_gen_Bnet_values >= self.gen_limits_Bnet_min_only[non_slack_gen_local_indices] - self.tolerance) and \
                               np.all(non_slack_gen_Bnet_values <= self.gen_limits_Bnet_max_only[non_slack_gen_local_indices] + self.tolerance)

        # Check SLACK BUS Net Injection Limits
        slack_bus_ok = True
        slack_Bnet_value = sol_Bnet_flat[self.slack_bus_index]
        slack_min_Bnet = self.gen_limits_Bnet_min_only[self.slack_bus_local_gen_idx]
        slack_max_Bnet = self.gen_limits_Bnet_max_only[self.slack_bus_local_gen_idx]
        slack_bus_ok = (slack_Bnet_value >= slack_min_Bnet - self.tolerance) and \
                       (slack_Bnet_value <= slack_max_Bnet + self.tolerance)

        # Check Load-Only Net Injection Limits - REMOVED (Fixed)

        # Check Power Balance
        bal_ok = abs(np.sum(sol_Bnet_flat)) < self.tolerance * self.num_buses

        # Feasible only if all checks pass
        feasible = line_ok and non_slack_gen_ok and slack_bus_ok and bal_ok

        if verbose or not feasible:
            print(f"--- Feasibility Check {'FAILED' if not feasible else 'PASSED'} (Min Cost Objective) ---")
            if not line_ok: print(f"  Line constraints failed.")
            else: print("  Line constraints met.")
            if not non_slack_gen_ok: print(f"  NON-SLACK Generator net injection limits failed (Buses: {self.non_slack_gen_indices+1}).")
            else: print("  NON-SLACK Generator net injection limits met.")
            if not slack_bus_ok: print(f"  SLACK BUS ({self.slack_bus_index+1}) net injection limits failed (Value={slack_Bnet_value:.4f}, Limits=[{slack_min_Bnet:.4f}, {slack_max_Bnet:.4f}]).")
            else: print(f"  SLACK BUS ({self.slack_bus_index+1}) net injection limits met.")
            # No load check message needed
            if not bal_ok: print(f"  Power balance check failed, sum(B_net)={np.sum(sol_Bnet_flat):.8f}")
            else: print("  Power balance met.")
            print("-" * 40)
        return feasible

    # MODIFIED _generate_random_solution: No Load Perturbation
    def _generate_random_solution(self):
        """
        Generates a new random solution by perturbing ONLY GENERATOR B_net values
        within their limits, then applying constraints (fixing loads, balancing via slack).
        """
        rand_sol_Bnet = self.initial_B_net.copy()
        n_gens = len(self.gen_indices)

        # Perturb ONLY Generator Net Injection within their Bnet limits
        if n_gens > 0:
            gen_Bnet_range = np.maximum(self.tolerance, self.gen_limits_Bnet_max_only - self.gen_limits_Bnet_min_only)
            perturbation_factor = 0.2
            perturbations_g = (np.random.rand(n_gens) - 0.5) * 2 * gen_Bnet_range * perturbation_factor
            # Apply perturbation and clip generators (including slack initially)
            rand_sol_Bnet[self.gen_indices] = np.clip(self.initial_B_net[self.gen_indices] + perturbations_g,
                                                      self.gen_limits_Bnet_min_only,
                                                      self.gen_limits_Bnet_max_only)

        # NO perturbation for load buses

        # Apply constraints (fixes loads, clips non-slack gens, balances via slack)
        return self._apply_constraints(rand_sol_Bnet)

    # MODIFIED _apply_local_search: Perturb Only Generators
    def _apply_local_search(self, solution_Bnet, iteration, max_iterations):
        """
        Applies local search by perturbing ONLY GENERATOR buses.
        """
        current_solution_Bnet = solution_Bnet.copy()
        current_fitness = self._calculate_fitness(current_solution_Bnet)
        if not np.isfinite(current_fitness): return solution_Bnet

        progress_ratio = iteration / max_iterations
        step_scale_factor = 0.1 * (1.0 - progress_ratio) + 0.01 * progress_ratio
        # Try perturbing a few times, focusing only on generators
        num_attempts = min(3, len(self.gen_indices)) # Fewer attempts, only on generators

        if len(self.gen_indices) == 0: return solution_Bnet # Cannot perturb if no generators

        for _ in range(num_attempts):
            # Choose a random GENERATOR bus to perturb
            idx_to_perturb = random.choice(self.gen_indices)

            perturbed_solution_Bnet = current_solution_Bnet.copy()
            perturb_range = 1.0

            # Determine perturbation range based on the chosen generator's Bnet limits
            gen_idx_local = np.where(self.gen_indices == idx_to_perturb)[0][0]
            op_range = self.gen_limits_Bnet_max_only[gen_idx_local] - self.gen_limits_Bnet_min_only[gen_idx_local]
            perturb_range = max(self.tolerance, op_range) * step_scale_factor

            perturbation = (random.random() - 0.5) * 2 * perturb_range
            perturbed_solution_Bnet[idx_to_perturb, 0] += perturbation

            # Apply constraints (fixes loads, clips non-slack, balances via slack)
            refined_solution_Bnet = self._apply_constraints(perturbed_solution_Bnet)
            refined_fitness = self._calculate_fitness(refined_solution_Bnet)

            if np.isfinite(refined_fitness) and refined_fitness < current_fitness:
                current_solution_Bnet = refined_solution_Bnet
                current_fitness = refined_fitness

        return current_solution_Bnet

    # --- Optimization Loop (optimize method) ---
    # NO CHANGE NEEDED in the main loop structure, adaptive logic, stagnation logic.
    # It relies on the modified _calculate_fitness, _apply_constraints etc.
    # The printed output within the loop should be updated to show cost.
    def optimize(self, B_unused, iterations=200, population_size=30):
        """ Main optimization loop - structure unchanged, uses modified helper methods. """
        print(f"Starting Optimization (Minimize Rescheduling Cost): Pop={population_size}, Iter={iterations}")
        initial_state_constrained = self._apply_constraints(self.initial_B_net)
        initial_population = [self._generate_random_solution() for _ in range(population_size)]
        initial_population[0] = initial_state_constrained

        fitness_values = [self._calculate_fitness(sol) for sol in initial_population]

        best_idx = np.argmin(fitness_values) if np.any(np.isfinite(fitness_values)) else -1
        if best_idx != -1:
            best_solution_Bnet = initial_population[best_idx].copy()
            best_fitness = fitness_values[best_idx] # best_fitness now represents cost + penalties
        else:
            print("ERROR: No finite fitness solutions found in initial population.")
            try: C_initial = np.dot(self.A, self.initial_B_net)
            except Exception: C_initial = np.full((self.num_lines, 1), np.nan)
            return self.initial_B_net.reshape(-1, 1), [], C_initial

        # Initial fitness is primarily cost + penalties
        initial_cost = self._get_rescheduling_cost(best_solution_Bnet)
        print(f"Initial Best Fitness (Cost+Penalties): {best_fitness:.4e}, Initial Rescheduling Cost: {initial_cost:.2f}")
        fitness_history = [best_fitness]

        algorithm_names = ['OOA', 'KHA', 'SHO']
        algo_success_count = {name: 0 for name in algorithm_names}; algo_attempts_count = {name: 0 for name in algorithm_names}
        algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}; adaptation_rate = 0.05
        stagnation_counter = 0; last_best_fitness = best_fitness

        for iteration in range(iterations):
            new_population = []; current_fitness_values = []
            # --- Adaptive Hybridization Logic (Unchanged) ---
            total_attempts = sum(algo_attempts_count.values())
            if total_attempts > 0:
                success_rates = {name: algo_success_count[name] / algo_attempts_count[name] if algo_attempts_count[name] > 0 else 0 for name in algorithm_names}
                total_rate = sum(success_rates.values())
                if total_rate > 1e-6:
                    target_probabilities = {name: rate / total_rate for name, rate in success_rates.items()}
                    for name in algorithm_names: algo_probabilities[name] = (1 - adaptation_rate) * algo_probabilities[name] + adaptation_rate * target_probabilities[name]
                    prob_sum = sum(algo_probabilities.values());
                    if prob_sum > 1e-6: algo_probabilities = {name: p / prob_sum for name, p in algo_probabilities.items()}
                    else: algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}
                current_weights = [algo_probabilities[name] for name in algorithm_names]
            else: current_weights = [1.0/len(algorithm_names)] * len(algorithm_names)
            # --- --- ---

            for i in range(population_size):
                try: algorithm = random.choices(algorithm_names, weights=current_weights, k=1)[0]
                except ValueError: algorithm = random.choice(algorithm_names)
                algo_attempts_count[algorithm] += 1
                current_solution_Bnet = initial_population[i]

                if algorithm == 'OOA': candidate_Bnet = self._apply_orcas_optimization(initial_population, i, best_solution_Bnet, iteration, iterations)
                elif algorithm == 'KHA': candidate_Bnet = self._apply_krill_herd(initial_population, i, best_solution_Bnet, iteration, iterations)
                else: candidate_Bnet = self._apply_spotted_hyena(initial_population, i, best_solution_Bnet, iteration, iterations)

                refined_candidate_Bnet = self._apply_local_search(candidate_Bnet, iteration, iterations)
                new_solution_Bnet = self._apply_constraints(refined_candidate_Bnet) # Fixes loads, balances via slack
                new_population.append(new_solution_Bnet)
                new_fitness = self._calculate_fitness(new_solution_Bnet) # Includes cost objective
                current_fitness_values.append(new_fitness)

                if np.isfinite(new_fitness) and new_fitness < best_fitness:
                    best_fitness = new_fitness; best_solution_Bnet = new_solution_Bnet.copy()
                    algo_success_count[algorithm] += 1

            initial_population = new_population; fitness_values = current_fitness_values
            if np.isfinite(best_fitness): fitness_history.append(best_fitness)

            # --- Stagnation & Diversification Logic (Unchanged) ---
            if abs(best_fitness - last_best_fitness) < self.tolerance * max(1.0, abs(last_best_fitness)): stagnation_counter += 1
            else: stagnation_counter = 0; last_best_fitness = best_fitness
            if stagnation_counter >= self.stagnation_threshold:
                print(f"\nStagnation detected at iteration {iteration}. Diversifying population...")
                num_to_replace = int(population_size * self.diversification_fraction)
                worst_indices = np.argsort(fitness_values)[-num_to_replace:]
                for idx in worst_indices:
                    initial_population[idx] = self._generate_random_solution() # Perturbs only generators
                    fitness_values[idx] = self._calculate_fitness(initial_population[idx])
                best_idx = np.argmin(fitness_values) if np.any(np.isfinite(fitness_values)) else -1
                if best_idx != -1:
                     best_solution_Bnet = initial_population[best_idx].copy()
                     best_fitness = fitness_values[best_idx]
                last_best_fitness = best_fitness; stagnation_counter = 0
                print(f"Diversification complete. New best fitness: {best_fitness:.4e}")
            # --- --- ---

            # Update printed output to show cost
            if iteration % 100 == 0 or iteration == iterations - 1:
                current_cost = self._get_rescheduling_cost(best_solution_Bnet)
                print(f"Iter {iteration}/{iterations}: BestFit(Cost+Pen)={best_fitness:.4e}, ReschedCost={current_cost:.2f} (Stag: {stagnation_counter}/{self.stagnation_threshold})")

        best_solution_Bnet = self._apply_constraints(best_solution_Bnet) # Final constraint check

        # --- Final Reporting ---
        total_attempts_final = sum(algo_attempts_count.values())
        if total_attempts_final > 0: print("\nAlgorithm Contributions (Attempts):", {k: f"{v/total_attempts_final*100:.1f}%" for k, v in algo_attempts_count.items()})
        final_rescheduling_cost = self._get_rescheduling_cost(best_solution_Bnet)
        print(f"\nFinal Minimum Rescheduling Cost: {final_rescheduling_cost:.2f} $/hr")
        # Deviation is no longer a primary objective, but can be calculated for info
        final_deviation = self._get_generator_deviation(best_solution_Bnet)
        print(f"Final Sum of Absolute Changes (Generators Only - Bnet): {final_deviation:.4f}")

        try: final_C = np.dot(self.A, best_solution_Bnet)
        except Exception: final_C = np.full((self.num_lines, 1), np.nan)
        return best_solution_Bnet, fitness_history, final_C

    # --- Memory Functions (Unchanged) ---
    def _check_memory(self, B_unused):
        if not self.memory: return None
        print("Checking memory for feasible solutions...")
        for _, stored_Bnet in reversed(self.memory):
            if self._is_feasible(stored_Bnet): # Uses updated feasibility check
                print("Using feasible B_net solution from memory.")
                return stored_Bnet
        print("No suitable feasible solution found in memory.")
        return None

    def _store_in_memory(self, initial_Bnet_state, optimized_Bnet_state):
         if self._is_feasible(optimized_Bnet_state): # Uses updated feasibility check
            self.memory.append((initial_Bnet_state.copy(), optimized_Bnet_state.copy()))
            print(f"Feasible B_net solution stored. Memory size: {len(self.memory)}")
         else:
            print("Optimized solution is infeasible, not storing in memory.")

    # --- Metaheuristic Implementations (OOA, KHA, SHO - Unchanged) ---
    def _apply_orcas_optimization(self,p,i,b,it,m_it):
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();a=2*(1-(it/m_it)**2);r1,r2=random.random(),random.random();
        if r1<0.5: d=np.abs(b_f-s_f);l=2*r2-1;d=np.maximum(d,1e-9);step=a*r2*(b_f-s_f);n_f=s_f+step
        else: idx=[j for j in range(len(p)) if j!=i];X=p[random.choice(idx)].flatten() if idx else s_f;A=2*a*r1-a;C=2*r2;D=np.abs(C*X-s_f);n_f=X-A*D
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2); return n_f.reshape(-1,1)
    def _apply_krill_herd(self,p,i,b,it,m_it):
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();Dmax=0.005*(1-it/m_it);Vf=.02;Nmax=.01;Dt=1.0;
        Ni=Nmax*(b_f-s_f);Fi=Vf*(b_f-s_f);d=np.random.uniform(-1,1,s_f.shape);Di=Dmax*d; n_f=s_f+Dt*(Ni+Fi+Di);
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2); return n_f.reshape(-1,1)
    def _apply_spotted_hyena(self,p,i,b,it,m_it):
        s=p[i].copy();b_f=b.flatten();s_f=s.flatten();h=5-it*(5/m_it);B=2*random.random();E=2*h*random.random()-h;
        D_b=np.abs(B*b_f-s_f);X1=b_f-E*D_b;
        if abs(E)>=1: idx=[j for j in range(len(p)) if j!=i];r_h=p[random.choice(idx)].flatten() if idx else s_f;D_h=np.abs(B*r_h-s_f);n_f=r_h-E*D_h
        else: n_f=X1
        n_f=np.nan_to_num(n_f,nan=np.mean(s_f),posinf=np.max(s_f)*2,neginf=np.min(s_f)*2); return n_f.reshape(-1,1)



In [4]:
# Cell 3: Wrapper Function (Aligned with Paper's Objective)
# MODIFIED function signature: Use Inc/Dec Costs
def optimize_power_flow_congestion_mgt(A, initial_B_net, fixed_load_full,
                                      line_limits,
                                      gen_costs_inc_full, gen_costs_dec_full, # <-- NEW Cost Inputs
                                      gen_limits_min_full, gen_limits_max_full,
                                      gen_indices, load_indices,
                                      slack_bus_index,
                                      iterations=5000, population_size=100):
    """
    Wrapper function for minimizing generator rescheduling cost using a SLACK BUS.
    Loads are fixed. Aligned with the objective in the reference paper.

    Args:
        A (np.ndarray): System matrix.
        initial_B_net (np.ndarray): Initial NET bus injections.
        fixed_load_full (np.ndarray): Fixed load demand (Pl >= 0) for ALL buses.
        line_limits (np.ndarray): Absolute limits for line flows.
        gen_costs_inc_full (np.ndarray): Incremental cost bids ($/MW inc) for ALL buses.
        gen_costs_dec_full (np.ndarray): Decremental cost bids ($/MW dec) for ALL buses.
        gen_limits_min_full (np.ndarray): Min generation (Pg) limit for ALL buses.
        gen_limits_max_full (np.ndarray): Max generation (Pg) limit for ALL buses.
        gen_indices (list or np.ndarray): 0-based indices for generator buses.
        load_indices (list or np.ndarray): 0-based indices for fixed load buses.
        slack_bus_index (int): 0-based index of the designated slack bus.
        iterations (int): Number of optimization iterations.
        population_size (int): Number of solutions in the population.

    Returns:
        tuple: (B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details)
    """
    print("--- Initializing Optimizer (Minimize Rescheduling Cost, SLACK BUS) ---")
    try:
        if 'HybridPowerFlowOptimizer' not in globals():
            raise NameError("Optimizer class 'HybridPowerFlowOptimizer' is not defined.")

        # Instantiate the optimizer with new cost arguments
        optimizer = HybridPowerFlowOptimizer(A, line_limits,
                                             gen_costs_inc_full, gen_costs_dec_full, # Pass inc/dec costs
                                             gen_limits_min_full, gen_limits_max_full,
                                             initial_B_net, fixed_load_full,
                                             gen_indices, load_indices,
                                             slack_bus_index)

    except (ValueError, NameError, IndexError) as e:
        print(f"ERROR initializing optimizer: {e}")
        try: C_unoptimized_calc = np.dot(A, initial_B_net)
        except Exception: C_unoptimized_calc = np.full((A.shape[0], 1), np.nan)
        return initial_B_net, [], C_unoptimized_calc, C_unoptimized_calc, False, {}

    # Get the potentially adjusted initial B_net from the optimizer
    initial_Bnet_from_opt = optimizer.initial_B_net.reshape(-1, 1)
    # Calculate flows for this initial state
    try: C_unoptimized = np.dot(A, initial_Bnet_from_opt)
    except Exception as e:
        print(f"Warning: Could not calculate unoptimized flows: {e}")
        C_unoptimized = np.full((A.shape[0], 1), np.nan)

    print("\n--- Checking Initial State Feasibility (Optimizer's Initial B_net) ---")
    initial_feasible = optimizer._is_feasible(initial_Bnet_from_opt, verbose=True) # Uses updated check
    print(f"Optimizer's initial state feasible: {initial_feasible}")
    if not initial_feasible:
        print("WARNING: Optimizer starting from an infeasible state.")

    print("\n--- Starting Optimization (Minimize Rescheduling Cost) ---")
    # Run optimization
    B_opt_net, fit_hist, C_opt = optimizer.optimize(None, iterations=iterations, population_size=population_size)

    print("\n--- Checking Final Solution Feasibility ---")
    final_feasible = optimizer._is_feasible(B_opt_net, verbose=True) # Uses updated check
    print(f"\nFinal feasibility: {final_feasible}")

    # Store details
    details = {
        "gen_indices": optimizer.gen_indices,
        "load_indices": optimizer.load_indices,
        "fixed_load": optimizer.fixed_load,
        "slack_bus_index": optimizer.slack_bus_index,
        "gen_costs_inc_only": optimizer.gen_costs_inc_only, # Store costs used
        "gen_costs_dec_only": optimizer.gen_costs_dec_only
    }

    return B_opt_net, fit_hist, C_opt, C_unoptimized, final_feasible, details


In [7]:
# Cell 4: Visualization Function (Aligned with Paper's Objective)
def visualize_results(A, B, B_optimized, C_optimized, C_unoptimized, line_limits,
                      gen_costs_inc_full, gen_costs_dec_full, # Use full cost arrays for summary
                      gen_limits_min_full, gen_limits_max_full,
                      gen_indices, load_indices, slack_bus_index, fixed_load, # Need fixed load for summary
                      fitness_history, final_feasible): # Pass feasibility status
    """
    Visualizes the optimization results (Minimize Rescheduling Cost Model).
    Loads are fixed.

    Args:
        # ... (previous args) ...
        gen_costs_inc_full (np.ndarray): Full array of incremental costs.
        gen_costs_dec_full (np.ndarray): Full array of decremental costs.
        fixed_load (np.ndarray): Fixed load array (num_buses,) used by optimizer.
        final_feasible (bool): Feasibility status of the optimized solution.
    """
    num_lines = A.shape[0]
    num_buses = A.shape[1]
    tolerance = 1e-6

    print("\n--- Generating Plots (Minimize Rescheduling Cost - Slack Bus) ---")

    # Plot 1: Line Flows Comparison (Code identical to previous version)
    try:
        plt.figure(figsize=(12, 6)); idx = np.arange(1, num_lines + 1)
        if isinstance(C_unoptimized, np.ndarray) and C_unoptimized.size == num_lines and np.all(np.isfinite(C_unoptimized)):
            plt.plot(idx, C_unoptimized.flatten(), 'o-', label='Initial Flows', alpha=0.7, markersize=4)
        else: print("Warning: Skipping initial flows plot (invalid data).")
        if isinstance(C_optimized, np.ndarray) and C_optimized.size == num_lines and np.all(np.isfinite(C_optimized)):
            plt.plot(idx, C_optimized.flatten(), 's--', label='Optimized Flows', alpha=0.9, markersize=5)
            flows_opt_abs = np.abs(C_optimized.flatten()); violations = np.where(flows_opt_abs > line_limits + tolerance)[0]
            if len(violations) > 0: plt.scatter(idx[violations], C_optimized.flatten()[violations], c='magenta', s=100, zorder=5, label=f'Violations ({len(violations)})', marker='x')
        else: print("Warning: Skipping optimized flows plot (invalid data).")
        plt.plot(idx, line_limits, 'r:', alpha=0.8, label='Limit (+)')
        plt.plot(idx, -line_limits, 'r:', alpha=0.8, label='Limit (-)')
        plt.xlabel('Line Index'); plt.ylabel('Power Flow (MW or p.u.)')
        plt.title(f'Line Flows Comparison (Slack Bus: {slack_bus_index+1})')
        plt.legend(); plt.grid(True, linestyle=':'); plt.xticks(idx[::max(1, num_lines//20)])
        plt.tight_layout(); plt.show()
    except Exception as e: print(f"Plotting error (Line Flows): {e}")

    # Plot 2: Fitness History (Code identical to previous version)
    try:
        if isinstance(fitness_history, (list, np.ndarray)) and len(fitness_history) > 1:
            plt.figure(figsize=(10, 5))
            finite_fitness = [f for f in fitness_history if np.isfinite(f)]
            if len(finite_fitness) > 1:
                plt.plot(finite_fitness, '.-', color='royalblue', label='Best Fitness (Cost+Penalties)', markersize=3, linewidth=1) # Label updated
                plt.xlabel('Iteration'); plt.ylabel('Fitness Value (Log Scale)')
                plt.title('Optimization Convergence'); plt.yscale('log')
                plt.legend(); plt.grid(True, linestyle=':'); plt.tight_layout(); plt.show()
            else: print("Warning: Not enough finite fitness values to plot convergence.")
        else: print("Warning: Skipping fitness plot (insufficient history data).")
    except Exception as e: print(f"Plotting error (Fitness History): {e}")

    # Plot 3: Bus Net Injections (B_net) Comparison (Code identical to previous version)
    try:
        plt.figure(figsize=(14, 7)); bus_idx_plot = np.arange(1, num_buses + 1); bar_width = 0.35
        colors_initial = []; colors_optimized = []
        for i in range(num_buses):
             if i == slack_bus_index: colors_initial.append('darkred'); colors_optimized.append('red')
             elif i in gen_indices: colors_initial.append('darkblue'); colors_optimized.append('darkgreen')
             else: colors_initial.append('skyblue'); colors_optimized.append('lightgreen')
        valid_B = isinstance(B, np.ndarray) and B.size == num_buses and np.all(np.isfinite(B))
        valid_B_opt = isinstance(B_optimized, np.ndarray) and B_optimized.size == num_buses and np.all(np.isfinite(B_optimized))
        if valid_B: plt.bar(bus_idx_plot - bar_width/2, B.flatten(), width=bar_width, label='Initial B_net (Slack=Red)', alpha=0.7, color=colors_initial)
        else: print("Warning: Skipping initial B_net plot (invalid data).")
        if valid_B_opt: plt.bar(bus_idx_plot + bar_width/2, B_optimized.flatten(), width=bar_width, label='Optimized B_net (Slack=Red)', alpha=0.8, color=colors_optimized)
        else: print("Warning: Skipping optimized B_net plot (invalid data).")
        plt.xlabel('Bus Index'); plt.ylabel('Net Power Injection (B_net = Pg - Pl)')
        plt.title(f'Bus Net Power Injections: Initial vs. Optimized (Slack Bus: {slack_bus_index+1})')
        plt.xticks(bus_idx_plot[::max(1, num_buses//20)]); plt.legend()
        plt.grid(True, axis='y', linestyle=':'); plt.axhline(0, color='black', linewidth=0.5)
        plt.tight_layout(); plt.show()
    except Exception as e: print(f"Plotting error (Bus Injections): {e}")

    # Plot 4: Generator Changes Plot (Code identical to previous version)
    try:
        if valid_B and valid_B_opt:
            changes = B_optimized.flatten() - B.flatten()
            bus_idx_plot = np.arange(1, num_buses + 1)
            if len(gen_indices) > 0:
                plt.figure(figsize=(10, 4)); gen_changes = changes[gen_indices]; gen_labels = bus_idx_plot[gen_indices]
                colors_gen_change = []
                for idx in gen_indices:
                    if idx == slack_bus_index: colors_gen_change.append('red' if gen_changes[np.where(gen_indices==idx)[0][0]] >= 0 else 'darkred')
                    else: colors_gen_change.append('forestgreen' if gen_changes[np.where(gen_indices==idx)[0][0]] >= 0 else 'firebrick')
                bar_indices_gen = np.arange(len(gen_indices))
                plt.bar(bar_indices_gen, gen_changes, color=colors_gen_change)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Generator Bus Index'); plt.ylabel('Change in Net Injection (ΔB_net = ΔPg)')
                plt.title(f'Generator Net Injection Changes (Optimized - Initial) (Slack: {slack_bus_index+1})')
                plt.xticks(bar_indices_gen, gen_labels); plt.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
            # Plot 5: Load Changes (Shedding) Plot - REMOVED (No shedding)
        else:
            print("Warning: Skipping generator change plot (invalid B or B_optimized data).")
    except Exception as e: print(f"Plotting error (Generator Changes): {e}")


    # --- Text Summary ---
    print("\n" + "="*30 + " RESULTS SUMMARY (Minimize Rescheduling Cost) " + "="*30)
    np.set_printoptions(precision=4, suppress=True)
    try:
        final_resched_cost = np.nan
        total_gen_change_abs = np.nan # For info

        # Calculate final metrics directly using B_net changes
        if valid_B and valid_B_opt and len(gen_indices) > 0:
             # Get the relevant costs
             costs_inc_only = gen_costs_inc_full[gen_indices]
             costs_dec_only = gen_costs_dec_full[gen_indices]
             # Calculate B_net change (equals Pg change)
             delta_Bnet = B_optimized.flatten()[gen_indices] - B.flatten()[gen_indices]
             current_cost = 0.0
             for i in range(len(gen_indices)):
                 delta = delta_Bnet[i]
                 if delta > tolerance: current_cost += costs_inc_only[i] * delta
                 elif delta < -tolerance: current_cost += costs_dec_only[i] * abs(delta)
             final_resched_cost = current_cost
             total_gen_change_abs = np.sum(np.abs(delta_Bnet))


        print(f"\nInitial B_net (Used by Optimizer):\n{B.flatten() if valid_B else 'N/A'}")
        print(f"\nOptimized B_net:\n{B_optimized.flatten() if valid_B_opt else 'N/A'}")
        if valid_B_opt: print(f"Sum Optimized B_net: {np.sum(B_optimized):.6f}")

        print(f"\nInitial Flows (C_unopt):\n{C_unoptimized.flatten() if isinstance(C_unoptimized, np.ndarray) and C_unoptimized.size == num_lines else 'N/A'}")
        print(f"\nOptimized Flows (C_opt):\n{C_optimized.flatten() if isinstance(C_optimized, np.ndarray) and C_optimized.size == num_lines else 'N/A'}")
        print("-" * 70)
        print("\nObjective Metrics:")
        print(f"  Final Generator Rescheduling Cost: {final_resched_cost:.2f} $/hr")
        print(f"  Total Absolute Generator Change (|ΔPg|): {total_gen_change_abs:.4f} MW") # Informational
        print(f"  Load Shedding: Not Applicable (Loads Fixed)")

        print("\nConstraint Check Summary:")
        print(f"  Final solution feasible: {'YES' if final_feasible else 'NO'}")

    except Exception as e:
        print(f"Error generating summary text: {e}")
    finally:
        np.set_printoptions(precision=8, suppress=False)

    print("=" * 70)
    print("Note: Objective is to minimize rescheduling cost. Loads are fixed.")
    print(f"Slack Bus used for balancing: {slack_bus_index+1}")
    print("=" * 70)


In [ ]:
# Cell 5: Main Execution Block (Aligned with Paper's Objective)
if __name__ == "__main__":

    # --- Ensure functions/classes are defined ---
    if 'HybridPowerFlowOptimizer' not in globals(): print("FATAL ERROR: HybridPowerFlowOptimizer class not defined."); exit()
    if 'optimize_power_flow_congestion_mgt' not in globals(): print("FATAL ERROR: optimize_power_flow_congestion_mgt function not defined."); exit() # Renamed wrapper
    if 'visualize_results' not in globals(): print("FATAL ERROR: visualize_results function not defined."); exit()

    # --- Load System Matrix A ---
    matrix_file_name = 'reshaped_data.csv'
    try:
        print(f"Loading system matrix A from: {matrix_file_name}")
        df = pd.read_csv(matrix_file_name, header=None); A = df.to_numpy()
        print(f"Successfully loaded matrix A with shape {A.shape}")
        if A.ndim != 2 or A.shape[0] == 0 or A.shape[1] == 0: raise ValueError("Invalid matrix A.")
    except FileNotFoundError: print(f"FATAL ERROR: Matrix file '{matrix_file_name}' not found."); exit()
    except Exception as e: print(f"FATAL ERROR loading matrix A: {e}. Exiting."); exit()

    num_lines_main = A.shape[0]; num_buses_main = A.shape[1]
    print(f"System dimensions: {num_lines_main} lines, {num_buses_main} buses.")
    all_bus_indices_set = set(range(num_buses_main))

    # --- Main Scenario Loop ---
    while True:
        print("\n" + "="*25 + " New Scenario (Minimize Rescheduling Cost) " + "="*25)
        print("Objective: Minimize Generator Rescheduling Cost (Inc/Dec Bids)")
        print("(Loads are Fixed, SLACK BUS Balancing)") # Update description

        try:
            # --- Get User Inputs ---

            # 1. Get Generator Indices (1-based)
            temp_gen_indices_0based = []
            while True:
                try:
                    gen_indices_str = input(f"\n>>> Enter GENERATOR bus numbers (1 to {num_buses_main}, space-separated): ")
                    user_gen_indices_list = [int(x) for x in gen_indices_str.split()]
                    valid_indices = True; gen_indices_set = set(); temp_gen_indices_0based_current = []
                    for idx_1based in user_gen_indices_list:
                        if 1 <= idx_1based <= num_buses_main:
                            idx_0based = idx_1based - 1
                            if idx_0based in gen_indices_set: print(f"  Error: Duplicate index {idx_1based}."); valid_indices = False; break
                            gen_indices_set.add(idx_0based); temp_gen_indices_0based_current.append(idx_0based)
                        else: print(f"  Error: Invalid bus number {idx_1based}."); valid_indices = False; break
                    if valid_indices: temp_gen_indices_0based = sorted(temp_gen_indices_0based_current); break
                except ValueError: print("  Error: Invalid input format."); continue # Loop back
                except EOFError: raise

            if not temp_gen_indices_0based: print("ERROR: At least one generator needed."); continue

            # 2. Get Slack Bus Index (1-based)
            slack_bus_index_input = -1
            while True:
                try:
                    slack_bus_1based_str = input(f">>> Enter SLACK BUS number (must be one of {np.array(temp_gen_indices_0based) + 1}): ")
                    slack_bus_1based = int(slack_bus_1based_str)
                    slack_bus_index_input_candidate = slack_bus_1based - 1
                    if slack_bus_index_input_candidate in temp_gen_indices_0based:
                        slack_bus_index_input = slack_bus_index_input_candidate; print(f"-> Using Bus {slack_bus_1based} as Slack Bus."); break
                    else: print(f"  Error: Slack bus {slack_bus_1based} is not in the list of generators.")
                except ValueError: print("  Error: Invalid number format."); continue
                except EOFError: raise

            # 3. Determine load-only indices
            temp_load_indices_0based = sorted(list(all_bus_indices_set - set(temp_gen_indices_0based)))
            print(f"-> Using Generators: {np.array(temp_gen_indices_0based) + 1}")
            print(f"-> Using Fixed Load Buses: {np.array(temp_load_indices_0based) + 1}") # Updated description

            # 4. Get Fixed Load (Pl) for ALL buses
            print(f"\n>>> Enter FIXED LOAD demand (Pl >= 0) for ALL buses ({num_buses_main} values):")
            fixed_load_input = np.zeros(num_buses_main)
            while True:
                try:
                    pl_str = input("  Fixed Loads (Pl): ")
                    pl_list = [float(x) for x in pl_str.split()]
                    if len(pl_list) == num_buses_main:
                        fixed_load_input = np.array(pl_list)
                        if np.any(fixed_load_input < 0): print("  Warning: Negative loads entered. Clamping to zero."); fixed_load_input = np.maximum(0, fixed_load_input)
                        break
                    else: print(f"  Error: Expected {num_buses_main} values, got {len(pl_list)}.")
                except ValueError: print("  Error: Invalid number format."); continue
                except EOFError: raise

            # 5. Get Initial Generation (Pg_initial) ONLY for GENERATOR buses
            pg_initial_input = np.zeros(num_buses_main)
            print(f"\n>>> Enter INITIAL GENERATION (Pg) ONLY for GENERATOR buses {np.array(temp_gen_indices_0based) + 1}:")
            for i in temp_gen_indices_0based:
                while True:
                    try: pg_initial_input[i] = float(input(f"    G{i+1} Initial Pg: ")); break
                    except ValueError: print("    Invalid number."); continue
                    except EOFError: raise

            # 6. Calculate Initial NET Injection (B = Pg - Pl)
            # IMPORTANT: Ensure load bus B_net is exactly -Pl initially
            B_input_net = (pg_initial_input - fixed_load_input)
            B_input_net[temp_load_indices_0based] = -fixed_load_input[temp_load_indices_0based] # Enforce fixed load B_net
            B_input_net = B_input_net.reshape(-1, 1) # Reshape for optimizer
            print("\nCalculated Initial Net Injections (B = Pg - Pl, Loads Fixed):")
            for i in range(num_buses_main): print(f"  Bus {i+1}: {B_input_net[i,0]:.4f}")

            # 7. Get Line Limits
            print(f"\n>>> Enter LINE power limits ({num_lines_main} values):")
            line_limits_input = np.zeros(num_lines_main)
            while True:
                try:
                    limits_str = input("  Line limits: "); line_limits_list = [float(x) for x in limits_str.split()]
                    if len(line_limits_list) == num_lines_main: line_limits_input = np.abs(np.array(line_limits_list)); break
                    else: print(f"  Error: Expected {num_lines_main} values."); continue
                except ValueError: print("  Error: Invalid number format."); continue
                except EOFError: raise

            # 8. Get Generator-Specific Data (Inc/Dec Costs, Pg Limits)
            gen_costs_inc_full_input = np.zeros(num_buses_main) # Incremental Costs
            gen_costs_dec_full_input = np.zeros(num_buses_main) # Decremental Costs
            gen_limits_pg_min_input = np.zeros(num_buses_main)
            gen_limits_pg_max_input = np.zeros(num_buses_main)

            print(f"\n>>> Enter Costs/Limits ONLY for GENERATOR buses {np.array(temp_gen_indices_0based) + 1}:")
            # Get Incremental Costs
            print("  Enter Gen INCREMENTAL Cost Bids ($/MW increase):")
            for i in temp_gen_indices_0based:
                while True:
                    try: gen_costs_inc_full_input[i] = float(input(f"    G{i+1} Incr. Cost: ")); break
                    except ValueError: print("    Invalid number."); continue
                    except EOFError: raise
            # Get Decremental Costs
            print("  Enter Gen DECREMENTAL Cost Bids ($/MW decrease):")
            for i in temp_gen_indices_0based:
                while True:
                    try: gen_costs_dec_full_input[i] = float(input(f"    G{i+1} Decr. Cost: ")); break
                    except ValueError: print("    Invalid number."); continue
                    except EOFError: raise
            # Get Min Limits
            print("  Enter Gen MIN Generation Limit (Pg_min MW):")
            for i in temp_gen_indices_0based:
                while True:
                    try: gen_limits_pg_min_input[i] = float(input(f"    G{i+1} Pg_min: ")); break
                    except ValueError: print("    Invalid number."); continue
                    except EOFError: raise
            # Get Max Limits
            print("  Enter Gen MAX Generation Limit (Pg_max MW):")
            for i in temp_gen_indices_0based:
                while True:
                    try:
                        max_val = float(input(f"    G{i+1} Pg_max: "))
                        if max_val < gen_limits_pg_min_input[i]: print(f"    Error: Max Pg < Min Pg. Re-enter."); continue
                        gen_limits_pg_max_input[i] = max_val; break
                    except ValueError: print("    Invalid number."); continue
                    except EOFError: raise

        except EOFError: print("\nInput interrupted. Restarting..."); continue
        except Exception as e: print(f"\nError during input: {e}. Restarting..."); continue

        # --- Call Optimization ---
        print("\n" + "="*25 + " Running Optimization (Minimize Cost) " + "="*25)
        B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = (None, [], None, None, False, {})
        try:
            opt_iterations = 10000; opt_pop_size = 500
            start_time = time.time()
            # Call the updated wrapper function
            B_optimized_net, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = optimize_power_flow_congestion_mgt(
                A, B_input_net, fixed_load_input,
                line_limits_input,
                gen_costs_inc_full_input, gen_costs_dec_full_input, # Pass inc/dec costs
                gen_limits_pg_min_input, gen_limits_pg_max_input,
                temp_gen_indices_0based, temp_load_indices_0based,
                slack_bus_index_input,
                iterations=opt_iterations, population_size=opt_pop_size
            )
            end_time = time.time()
            print(f"Optimization Duration: {end_time - start_time:.2f} seconds")

        except NameError as e: print(f"FATAL ERROR: Function not defined ({e})."); break
        except Exception as e:
            print(f"Error during optimization: {e}")
            try:
                if input("Try another scenario anyway? (y/n):").lower() != 'y': break
                else: continue
            except EOFError: print("\nExiting..."); break

        print("\n" + "="*25 + " Optimization Finished " + "="*25)

        # --- Post-processing: Calculate Optimized Pg ---
        # (Load shed calculation is removed)
        final_fixed_load = opt_details.get("fixed_load", np.zeros(num_buses_main))
        final_gen_indices = opt_details.get("gen_indices", np.array([]))
        optimized_Pg = np.zeros(num_buses_main)
        if B_optimized_net is not None and final_feasible:
            B_opt_flat = B_optimized_net.flatten()
            if len(final_gen_indices) > 0:
                 optimized_Pg[final_gen_indices] = B_opt_flat[final_gen_indices] + final_fixed_load[final_gen_indices]

        # --- Visualize Results ---
        try:
            print("\nVisualizing results...")
            final_slack_index = opt_details.get("slack_bus_index", -1)
            final_load_indices = opt_details.get("load_indices", np.array([]))
            visualize_results(A, B_input_net, B_optimized_net, C_optimized, C_unoptimized, line_limits_input,
                              gen_costs_inc_full_input, gen_costs_dec_full_input, # Pass cost arrays
                              gen_limits_pg_min_input, gen_limits_pg_max_input,
                              final_gen_indices, final_load_indices, final_slack_index,
                              final_fixed_load, # Pass fixed load for summary context
                              fitness_history, final_feasible) # Pass feasibility status
        except NameError as e: print(f"Error: Visualization function not defined ({e}).")
        except Exception as e: print(f"Error during visualization: {e}")

        # --- Save Results ---
        if final_feasible and B_optimized_net is not None:
            try:
                save = input("\nSave detailed results? (y/n): ").lower()
                if save == 'y':
                    default_fname = "congestion_mgt_results.txt"
                    fname = input(f"Filename (default: {default_fname}): ").strip() or default_fname
                    print(f"Saving results to {fname}...")
                    try:
                        with open(fname, 'w') as f:
                            save_gen_indices = opt_details.get("gen_indices", np.array([]))
                            save_load_indices = opt_details.get("load_indices", np.array([]))
                            save_slack_idx = opt_details.get("slack_bus_index", -1)
                            final_resched_cost = np.nan

                            # Recalculate final cost for saving
                            if len(save_gen_indices) > 0:
                                costs_inc_only = gen_costs_inc_full_input[save_gen_indices]
                                costs_dec_only = gen_costs_dec_full_input[save_gen_indices]
                                delta_Bnet = B_optimized_net.flatten()[save_gen_indices] - B_input_net.flatten()[save_gen_indices]
                                current_cost = 0.0
                                for i in range(len(save_gen_indices)):
                                    delta = delta_Bnet[i]
                                    if delta > tolerance: current_cost += costs_inc_only[i] * delta
                                    elif delta < -tolerance: current_cost += costs_dec_only[i] * abs(delta)
                                final_resched_cost = current_cost

                            f.write("Congestion Management Results (Minimize Rescheduling Cost)\n")
                            f.write(f"SLACK BUS: Bus {save_slack_idx+1}\n" if save_slack_idx != -1 else "SLACK BUS: None\n")
                            f.write("="*30+"\n\n"); f.write(f"Matrix A (Shape: {A.shape}):\n"); np.savetxt(f, A, fmt='%.4f'); f.write("\n")
                            f.write("Line Limits:\n"); [f.write(f"L{i+1}: {l:.2f}\n") for i,l in enumerate(line_limits_input)]
                            f.write("\nFixed Loads (Pl):\n"); [f.write(f"Bus {i+1}: {pl:.4f}\n") for i,pl in enumerate(final_fixed_load)]
                            f.write("\nGen Incr. Costs ($/MW):\n"); [f.write(f"G{idx+1}: {gen_costs_inc_full_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nGen Decr. Costs ($/MW):\n"); [f.write(f"G{idx+1}: {gen_costs_dec_full_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nGen Min Generation (Pg_min):\n"); [f.write(f"G{idx+1}: {gen_limits_pg_min_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nGen Max Generation (Pg_max):\n"); [f.write(f"G{idx+1}: {gen_limits_pg_max_input[idx]:.2f}\n") for idx in save_gen_indices]
                            f.write("\nInitial Net Injection (B_net):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_input_net)]
                            f.write("\nOptimized Net Injection (B_net):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_optimized_net)]
                            f.write("\nOptimized Generation (Pg = B_net_opt + Pl):\n"); [f.write(f"Bus {idx+1}: {optimized_Pg[idx]:.4f}\n") for idx in save_gen_indices]
                            # No load shed to report
                            f.write("\nInitial Flows (C_unopt):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_unoptimized)]
                            f.write("\nOptimized Flows (C_opt):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_optimized)]
                            f.write(f"\nFinal Minimum Rescheduling Cost: {final_resched_cost:.2f} $/hr\n")
                            f.write(f"Feasible: {'YES' if final_feasible else 'NO'}\n")
                        print(f"Results saved to {fname}")
                    except Exception as e: print(f"ERROR saving results: {e}")
            except EOFError: print("\nInput interrupted."); continue # Continue to next scenario prompt
        elif not final_feasible: print("\nFinal solution infeasible. Results not saved.")
        else: print("\nOptimization did not produce a valid result. Nothing to save.")

        # --- Ask to run again ---
        try:
            if input("\nRun another scenario? (y/n):").lower() != 'y': print("\nExiting..."); break
            else: print("\nRestarting scenario...\n" + "-"*70)
        except EOFError: print("\nExiting..."); break

    print("\nScript finished.")



Loading system matrix A from: reshaped_data.csv
Successfully loaded matrix A with shape (41, 30)
System dimensions: 41 lines, 30 buses.

========================= New Scenario (Minimize Rescheduling Cost) =========================
Objective: Minimize Generator Rescheduling Cost (Inc/Dec Bids)
(Loads are Fixed, SLACK BUS Balancing)



>>> Enter GENERATOR bus numbers (1 to 30, space-separated):  1 2 5 8 11 13
>>> Enter SLACK BUS number (must be one of [ 1  2  5  8 11 13]):  1


-> Using Bus 1 as Slack Bus.
-> Using Generators: [ 1  2  5  8 11 13]
-> Using Fixed Load Buses: [ 3  4  6  7  9 10 12 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30]

>>> Enter FIXED LOAD demand (Pl >= 0) for ALL buses (30 values):


  Fixed Loads (Pl):  0.00 32.55 3.60 11.40 141.30 0.00 34.20 45.00 0.00 8.70 0.00 16.80 0.00 9.30 12.30 5.25 13.50 4.80 14.25 3.30 26.25 0.00 4.80 13.05 0.00 5.25 0.00 0.00 3.60 15.90



>>> Enter INITIAL GENERATION (Pg) ONLY for GENERATOR buses [ 1  2  5  8 11 13]:
